In [1]:
import numpy as np 
import pandas as pd 
import seaborn as sns
from epiweeks import Week
import matplotlib.pyplot as plt

In [2]:
def add_epiweek_label(df_w):
    '''
    This function assumes that the dataframe has a datetime index
    and add the epiweek and year value
    '''

    df_w['epiweek_label'] = [Week.fromdate(x) for x in df_w.index]

    df_w['epiweek_label'] = df_w['epiweek_label'].astype(str)

    df_w['epiweek'] = df_w['epiweek_label'].astype(str).str[-2:].astype(int)
    
    df_w['year'] = df_w['epiweek_label'].astype(str).str[:4].astype(int)

    return df_w


def load_agg_clima(state):
    '''
    This function load and aggregates the climatic variables by the epidemiological year
    '''
    df_climate = pd.read_parquet(f'./climate/{state}_climate.parquet', columns = ['geocodigo', 'temp_med', 'umid_med', 'precip_tot'])
    
    df_climate.index = pd.to_datetime(df_climate.index)
        
    df_climate = df_climate.loc[(df_climate.index >= '2010-01-01') & (df_climate.index<= '2019-12-31')].sort_index()
    
    df_climate = add_epiweek_label(df_climate)
    
    
    df_clima_agg = pd.DataFrame()
    
    for year in np.arange(2010, 2020):
    
        filter1 = (df_climate.year == year) & (df_climate.epiweek >= 41)
        filter2 = (df_climate.year == year+1) & (df_climate.epiweek < 41)
        
        df_ = df_climate.loc[filter1 | filter2].reset_index().groupby('geocodigo').agg({'temp_med': 'mean', 'umid_med': 'mean', 'precip_tot': 'sum'}).reset_index()
    
        df_['year'] = year
        
        df_clima_agg = pd.concat([df_clima_agg, df_])
    
    return df_clima_agg    

In [3]:
state = 'AC'

In [4]:
load_agg_clima(state)

,geocodigo,temp_med,umid_med,precip_tot,year
0,1200013,26.572070,81.623687,661.716230,2010
1,1200054,25.622488,81.759386,648.937363,2010
2,1200104,25.920781,82.781763,653.966545,2010
3,1200138,26.294518,83.267535,656.895924,2010
4,1200179,26.203526,82.579392,603.846421,2010
...,...,...,...,...,...
17,1200450,26.080407,85.655146,216.112200,2019
18,1200500,25.851864,86.834722,252.352120,2019
19,1200609,25.578399,86.976728,245.493299,2019
20,1200708,25.715247,87.492272,204.548860,2019


In [5]:
states_BR = ['AL',
 'BA',
 'CE',
 'MA',
 'PB',
 'PE',
 'PI',
 'SE',
 'RN',
 'SP',
 'MG',
 'RJ',
 'ES',
 'AM',
 'AP',
 'TO',
 'RR',
 'RO',
 'AC',
 'PA',
 'DF',
 'GO',
 'MT',
 'MS',
 'RS',
 'SC',
 'PR']


In [6]:
df_perfis = pd.read_csv('./Dengue/dengue_pattern_10_19.csv', sep = ';', usecols = ['muni_code', 'dengue_pattern'])

df_perfis.head()

,muni_code,dengue_pattern
0,1100015,Episódico/Epidêmico
1,1100023,Epidêmico
2,1100031,Episódico/Epidêmico
3,1100049,Epidêmico
4,1100056,Episódico/Epidêmico


The cell below concatenate the aggregated dataframes of all states in just one:

In [7]:
df_climate_all = pd.DataFrame()

for state in states_BR: 
    
    df_climate = load_agg_clima(state)
    
    df_climate_all = pd.concat([df_climate_all, df_climate])

# merge the climatic dataset with the dengue patterns 
df_climate_all = df_climate_all.merge(df_perfis, left_on = 'geocodigo', right_on = 'muni_code')
    
df_climate_all.head()

,geocodigo,temp_med,umid_med,precip_tot,year,muni_code,dengue_pattern
0,2700102,25.917411,71.160050,174.737289,2010,2700102,Episódico/Epidêmico
1,2700201,25.484200,80.407899,424.205669,2010,2700201,Episódico/Epidêmico
2,2700300,25.588142,77.216115,277.945870,2010,2700300,Epidêmico
3,2700409,24.897218,81.989589,423.138619,2010,2700409,Episódico/Epidêmico
4,2700508,25.800147,78.780638,491.421552,2010,2700508,Episódico


Verifying the presence of null values in the data: 

In [8]:
df_climate_all.isnull().sum()

geocodigo         0
temp_med          0
umid_med          0
precip_tot        0
year              0
muni_code         0
dengue_pattern    0
dtype: int64

Save the dataframe: 

In [9]:
%%time
df_climate_all.to_csv('./clima/clima_agg_10_19.csv')

CPU times: user 187 ms, sys: 15.5 ms, total: 202 ms
Wall time: 237 ms
